# Tool Calling Prompts

**Module:** 07-prompt-engineering

**Notebook:** `07-tool-calling-prompts.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Function Calling Basics** with clear contracts and failure modes
- Explain and apply **Prompting for Tool Use** with clear contracts and failure modes
- Explain and apply **API Tools** with clear contracts and failure modes
- Explain and apply **Database Tools** with clear contracts and failure modes
- Explain and apply **Search Tools** with clear contracts and failure modes
- Explain and apply **Calculator Tools** with clear contracts and failure modes
- Explain and apply **Custom Tools** with clear contracts and failure modes
- Explain and apply **Mini Tool Router** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Tool Calling Prompts

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Function Calling Basics**
2. **Prompting for Tool Use**
3. **API Tools**
4. **Database Tools**
5. **Search Tools**
6. **Calculator Tools**
7. **Custom Tools**
8. **Mini Tool Router**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Function Calling Basics

### Definition
**Function Calling Basics** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Function Calling Basics typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Function Calling Basics: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Function Calling Basics as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Function Calling Basics as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Function Calling Basics
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Function Calling Basics when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Function Calling Basics improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Function Calling Basics" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Function Calling Basics"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Prompting for Tool Use

### Definition
**Prompting for Tool Use** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Prompting for Tool Use typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Prompting for Tool Use: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Prompting for Tool Use as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Prompting for Tool Use as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Prompting for Tool Use
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Prompting for Tool Use when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Prompting for Tool Use" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Prompting for Tool Use"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Prompting for Tool Use

**Situation:** A team wants to productionize a feature involving **Prompting for Tool Use**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## API Tools

### Definition
**API Tools** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around API Tools typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For API Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain API Tools as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating API Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for API Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use API Tools when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "API Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "API Tools"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Database Tools

### Definition
**Database Tools** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Database Tools typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Database Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Database Tools as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Database Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Database Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Database Tools when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Database Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Database Tools"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Database Tools

**Situation:** A team wants to productionize a feature involving **Database Tools**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Search Tools

### Definition
**Search Tools** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Search Tools typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Search Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Search Tools as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Search Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Search Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Search Tools when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Search Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Search Tools"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Calculator Tools

### Definition
**Calculator Tools** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Calculator Tools typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Calculator Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Calculator Tools as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Calculator Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Calculator Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Calculator Tools when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Calculator Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Calculator Tools"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Calculator Tools

**Situation:** A team wants to productionize a feature involving **Calculator Tools**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Custom Tools

### Definition
**Custom Tools** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Custom Tools typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Custom Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Custom Tools as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Custom Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Custom Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Custom Tools when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Custom Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Custom Tools"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Mini Tool Router

### Definition
**Mini Tool Router** is a core building block in 07-tool-calling-prompts within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Mini Tool Router typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Mini Tool Router: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Mini Tool Router as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Mini Tool Router as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Mini Tool Router
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Mini Tool Router when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Mini Tool Router" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Mini Tool Router"
    notebook: str = "07-tool-calling-prompts"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_7 = ConceptContract()
print(json.dumps({"contract": asdict(contract_7), "health": contract_7.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Mini Tool Router

**Situation:** A team wants to productionize a feature involving **Mini Tool Router**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Tool Calling Prompts**.

| Topic | Do | Don't |
|-------|----|-------|
| Function Calling Basics | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Prompting for Tool Use | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| API Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Database Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Search Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Calculator Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Function Calling Basics | Key concept covered in this notebook; see its section for definition and pitfalls |
| Prompting for Tool Use | Key concept covered in this notebook; see its section for definition and pitfalls |
| API Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Database Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Search Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Calculator Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Custom Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Mini Tool Router | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Tool Calling Prompts** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **07-prompt-engineering**.


## Try It Yourself

1. Implement a failing test/fixture for **Function Calling Basics**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Prompting for Tool Use**, then fix your demo until it passes.
3. Implement a failing test/fixture for **API Tools**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Database Tools**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Search Tools**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
